# Demo 1 — Validate a merge before trusting it

**Learning objectives**

- State each input table's row grain and test its claimed keys.
- Predict cardinality and row preservation before merging.
- Use `validate=` and `indicator=True` to expose a duplicate dimension key and an orphan foreign key.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch and rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixtures contain invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"
try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None
if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd
print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Resolve and verify two prepared inputs

`visits` has grain one row per recorded visit. `sites_history` has grain one row per version of a site's metadata; therefore `site_code` is intentionally not unique until the supplied `record_status == 'current'` rule is applied. The bootstrap uses committed fixtures when present and otherwise recreates the same supplied bytes in runtime-local storage.

In [ ]:
from hashlib import sha256
from pathlib import Path

FIXTURES = {
    "visits.csv": {
        "relative": Path("06") / "demo" / "data" / "visits.csv",
        "sha256": "ccff0b9eaab1b6aae702734628db50b5223b04efc0071e1e9b4b9d6796e0c930",
        "bytes": (
            b"visit_id,participant_id,visit_number,site_code,status,measure\n"
            b"V001,P01,1,N,complete,12.5\n"
            b"V002,P01,2,N,complete,14.0\n"
            b"V003,P02,1,S,complete,9.5\n"
            b"V004,P03,1,W,complete,11.0\n"
            b"V005,P04,1,N,complete,13.5\n"
            b"V006,P05,1,X,complete,8.0\n"
        ),
    },
    "sites_history.csv": {
        "relative": Path("06") / "demo" / "data" / "sites_history.csv",
        "sha256": "42e3b766ca41024b33883463d49c9be56d3998536e7f00e0ba483dd799fdd935",
        "bytes": (
            b"site_code,site_name,region,record_status\n"
            b"N,North Clinic Legacy,north,retired\n"
            b"N,North Clinic,north,current\n"
            b"S,South Clinic,south,current\n"
            b"W,West Clinic,west,current\n"
        ),
    },
}


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


resolved = {}
for name, contract in FIXTURES.items():
    path = find_course_file(Path.cwd(), contract["relative"])
    if path is None:
        data_dir = Path.cwd() / "data"
        data_dir.mkdir(parents=True, exist_ok=True)
        path = data_dir / name
        path.write_bytes(contract["bytes"])
    assert sha256(path.read_bytes()).hexdigest() == contract["sha256"]
    resolved[name] = path

visits = pd.read_csv(
    resolved["visits.csv"],
    dtype={"visit_id": "string", "participant_id": "string", "site_code": "string", "status": "string"},
)
sites_history = pd.read_csv(
    resolved["sites_history.csv"],
    dtype={"site_code": "string", "site_name": "string", "region": "string", "record_status": "string"},
)
print(visits)
print(sites_history)

## Test key claims before the merge

`visit_id` is the visit primary key. (`participant_id`, `visit_number`) is another candidate key. `site_code` is a foreign key in `visits`. The history table fails a one-row-per-site key claim because it contains two versions of site N.

In [ ]:
assert visits["visit_id"].notna().all()
assert visits["visit_id"].is_unique
assert visits[["participant_id", "visit_number"]].notna().all().all()
assert not visits.duplicated(subset=["participant_id", "visit_number"]).any()
assert visits["site_code"].notna().all()

duplicate_site_rows = sites_history.loc[
    sites_history.duplicated(subset=["site_code"], keep=False)
].copy()
assert duplicate_site_rows["site_code"].tolist() == ["N", "N"]
duplicate_site_rows

## Predict cardinality and preservation

After selecting the documented current metadata rows, site codes may repeat on the visit side and must be unique on the site side: **many-to-one**. The preservation goal is to keep every visit, including a visit whose site metadata is absent, so the operation is a left merge and its expected row count is six.

In [ ]:
duplicate_contract_failed = False
try:
    visits.merge(
        sites_history,
        on="site_code",
        how="left",
        validate="many_to_one",
    )
except pd.errors.MergeError as error:
    duplicate_contract_failed = True
    print("Expected validation failure:", type(error).__name__)

assert duplicate_contract_failed

## Apply the supplied version rule, then merge

The source contract—not the merge itself—says `record_status == 'current'` identifies the active metadata row. Filter by that rule, retest the site key, and only then execute the predicted merge. `indicator=True` preserves unmatched-row evidence.

In [ ]:
sites_current = sites_history.loc[
    sites_history["record_status"].eq("current"),
    ["site_code", "site_name", "region"],
].copy()
assert sites_current["site_code"].notna().all()
assert sites_current["site_code"].is_unique

merge_audit = visits.merge(
    sites_current,
    on="site_code",
    how="left",
    validate="many_to_one",
    indicator=True,
)
unmatched_visits = merge_audit.loc[
    merge_audit["_merge"].eq("left_only"),
    ["visit_id", "site_code"],
].copy()
unmatched_visits

## Verify the preservation goal and diagnostics

A successful merge call is not enough. Verify the predicted row count, preserved primary keys, match counts, and exact orphan before using the result.

In [ ]:
source_counts = merge_audit["_merge"].value_counts().to_dict()
assert len(merge_audit) == len(visits) == 6
assert merge_audit["visit_id"].is_unique
assert set(merge_audit["visit_id"]) == set(visits["visit_id"])
assert source_counts["both"] == 5
assert source_counts["left_only"] == 1
assert source_counts["right_only"] == 0
assert unmatched_visits.to_dict(orient="records") == [{"visit_id": "V006", "site_code": "X"}]
assert merge_audit.loc[merge_audit["visit_id"].eq("V006"), "site_name"].isna().all()
print("Demo 1 validated merge diagnostics passed")
merge_audit